In [ ]:

import numpy as np
from sklearn.linear_model import LinearRegression



def Poly_8(X):
    P_0 = X**0
    P_1 = X
    P_2 = X**2
    P_3 = X**3
    P_4 = X**4
    P_5 = X**5
    P_6 = X**6
    P_7 = X**7
    P_8 = X**8
    return np.column_stack((P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8))

def Ker(x):
    return 2 * np.sin(np.arcsin(2 * x - 1) / 3)

def LSM_ISD(S0, K, r, sd, T, I, M, alpha, alpha_opt, n):
    
    prices = []
    deltas = []
    gammas = []
    for i in range(n):
        dt = T/M
        M=int(M*T)
        df = np.exp(-r*dt)
        S = np.zeros((I, M + 1))
        U = np.random.uniform(0, 1, I)
        S_init = S0 + alpha*Ker(U)  
        S[:, 0] = S_init
        S[:, 1:] = (S_init)[:, np.newaxis] * np.exp(np.cumsum((r - 0.5 * sd ** 2) * dt + sd * np.random.normal(0, np.sqrt(dt), size=(I, M)), axis=1))

    
        s = S/K 
        h = np.maximum(1 - s, 0)
        h[:, 0] = 0 
        h_copy = h.copy()

        for i in range(M, 1, -1):
            ITM = h[:, i - 1] > 0
            if np.sum(ITM) == 0:
                continue  

            Y = df * h_copy[ITM, i]
            X = s[ITM, i - 1]
            Z = Poly_8(X)

            reg1 = LinearRegression().fit(Z, Y)

            basis = Poly_8(s[:, i - 1])
            cont = reg1.predict(basis)

            h[:, i - 1] = np.where(h[:, i - 1] <= cont, 0, h[:, i - 1])
            h[h[:, i - 1] > 0, i:] = 0
            h_copy[:, i - 1] = np.where(h[:, i - 1] == 0, h_copy[:, i] * df, h_copy[:, i - 1])

        df_vec = np.power(df, np.arange(M + 1)-1).reshape(-1, 1)

        disc_CashFlows = h @ df_vec

        Z_1 = disc_CashFlows.flatten()
        
        X_poly = Poly_8(s[:,1]) 
        
        
        reg2 = LinearRegression().fit(X_poly, Z_1)

        value_t1 = reg2.predict(X_poly)

        value = np.maximum(value_t1, np.maximum(1-s[:,1],0))* df * K



        
        inside_opt_mask = (S[:, 0] - S0 >= -alpha_opt) & (S[:, 0] - S0 <= alpha_opt)

        value = value[inside_opt_mask]
        S = S[inside_opt_mask]


        X_poly2 = Poly_8(S[:, 0] - S0)
        reg3 = LinearRegression().fit(X_poly2, value)


        price = reg3.intercept_
        prices.append(price)

        delta = reg3.coef_[1]
        deltas.append(delta)

        gamma = 2* reg3.coef_[2]
        gammas.append(gamma)
    avg_price = np.mean(prices)    
    avg_delta = np.mean(deltas)
    avg_gamma = np.mean(gammas)

    price_std = np.std(prices, ddof=1)
    delta_std = np.std(deltas, ddof=1)
    gamma_std = np.std(gammas, ddof=1)


    return avg_price, price_std, avg_delta, avg_gamma, delta_std, gamma_std

import numpy as np

def CRR(S0, K, T, r, sd, N):
    dt = T / N
    u = np.exp(sd * np.sqrt(dt))
    d = np.exp(-sd * np.sqrt(dt))
    q = (np.exp(r * dt) - d) / (u - d)
    df = np.exp(-r * dt)

    # Step N stock prices
    S = S0 * d ** (np.arange(N, -1, -1)) * u ** (np.arange(0, N + 1, 1))
    C = np.maximum(K - S, 0)  # Put option payoff

    # Save prices and option values for Gamma calculation at step 2
    S_tree = {}
    C_tree = {}

    for i in np.arange(N - 1, -1, -1):
        S = S0 * d ** (np.arange(i, -1, -1)) * u ** (np.arange(0, i + 1, 1))
        continuation = df * (q * C[1:i + 2] + (1 - q) * C[0:i + 1])
        exercise = np.maximum(K - S, 0)
        C = np.maximum(continuation, exercise)

        # Store levels for Delta and Gamma calculation
        if i == 1:
            C_tree[1] = C.copy()
            S_tree[1] = S.copy()
        elif i == 2:
            C_tree[2] = C.copy()
            S_tree[2] = S.copy()

    # Delta ≈ (C_up - C_down) / (S_up - S_down)
    C_up = C_tree[1][0]
    C_down = C_tree[1][1]
    S_up = S_tree[1][0]
    S_down = S_tree[1][1]
    delta = (C_up - C_down) / (S_up - S_down)

    # Gamma ≈ ((C_uu - C_ud) / (S_uu - S_ud) - (C_ud - C_dd) / (S_ud - S_dd)) / ((S_uu - S_dd)/2)
    C_uu = C_tree[2][0]
    C_ud = C_tree[2][1]
    C_dd = C_tree[2][2]
    S_uu = S_tree[2][0]
    S_ud = S_tree[2][1]
    S_dd = S_tree[2][2]
    gamma = ((C_uu - C_ud) / (S_uu - S_ud) - (C_ud - C_dd) / (S_ud - S_dd)) / ((S_uu - S_dd) / 2)

    return round(C[0], 3), round(delta, 4), round(gamma, 4)





_, BM_delta, BM_gamma = CRR(S0=40, K=40, T=1, r=0.06, sd=0.2, N=10**3)
# print("Option Price:", price)
print("BM Delta:", BM_delta)
print("BM Gamma:", BM_gamma)


price, std, delta, gamma,delta_std,gamma_std = LSM_ISD(40,40,0.06,0.2,1,10**5,50,0.5,5,1)
print("Option Price:", round(price,3))
print("std", std)
print("Delta:", round(delta,4))
print("Delta Std", round(delta_std,4))
print("Delta dev", round((BM_delta-delta)/delta*100,3))
print("Gamma:", round(gamma,4))
print("Gamma Std", round(gamma_std,4))
print("Gamma dev", round((BM_gamma-gamma)/gamma*100,3))


